# 2.3 对应典型算子结构
不同算子的数据访问模式差异巨大：有的逐元素连续处理，有的做矩阵乘累加，有的按索引离散访存。950 的分离架构（Cube + Vector）与混合编程模型（SIMD + SIMT）正是为高效覆盖这些模式而设计。

本节将三类算子结构与硬件计算单元对应起来，为后续动手实践建立直觉。

本节内容：
- 连续类矢量算子（SIMD / Vector Core）
- 矩阵算子（Cube Core）
- 离散类矢量算子（SIMT / Vector Core）
- 算子结构与硬件单元映射总表


## 一、连续类矢量算子（SIMD / Vector Core）
### 1.1 代表算子：Add
数学表达式：$\vec{z} = \vec{x} + \vec{y}$，即 $z[i] = x[i] + y[i]$。

### 1.2 数据流
1. **搬入**：将输入 x、y 从 Global Memory 搬至 Vector Core 的 Unified Buffer（UB）。
2. **计算**：Vector Core 的 SIMD 指令一次处理整段连续数据，执行逐元素加法。
3. **搬回**：将结果 z 从 UB 搬回 Global Memory。

### 1.3 为什么用 SIMD
数据连续排列，SIMD 指令通过基地址+步长即可批量寻址，无需逐元素地址计算。950 的双发射 SIMD 可同时执行搬运和计算指令，实现流水重叠。

### 1.4 典型算子举例
Add、Sub、Mul、ReLU、GELU、Sigmoid、Tanh、Softmax、LayerNorm（向量部分）等。


## 二、矩阵算子（Cube Core）
### 2.1 代表算子：MatMul
数学表达式：$C = A \times B$，A∈[M,K]，B∈[K,N]，C∈[M,N]。

### 2.2 数据流
1. **搬入 L1**：将矩阵块从 GM 搬至 Cube Core 的 L1 Buffer。
2. **搬入 L0A/L0B**：从 L1 搬至 L0A/L0B（矩阵乘操作数缓冲）。
3. **Mmad 累加**：Cube 单元执行矩阵乘加指令（Mmad），结果累加至 L0C。
4. **搬回 GM**：将 L0C 结果搬回 Global Memory。

### 2.3 布局转换
Cube 计算要求数据按 **NZ/ZN 布局**排布（分块矩阵存储格式），而 GM 中通常是 ND 布局。因此搬运过程中需自动完成 ND→NZ/ZN 的布局转换。950 的 NDDMA 可在搬运时硬化完成这一转换。

### 2.4 典型算子举例
MatMul、Conv2D（im2col 后转化为矩阵乘）、BatchMatMul、FlashAttention（Cube 部分）等。


## 三、离散类矢量算子（SIMT / Vector Core）
### 3.1 代表算子：Gather
数学表达式：`output[i] = input[index[i]]`，每个输出元素的源地址由 index 数组决定，彼此独立且不连续。

### 3.2 数据流
1. **线程分配**：为每个输出元素分配一个 SIMT 线程。
2. **独立寻址**：各线程按 `index[i]` 直接从 GM 读取 `input` 中的一个元素。
3. **独立写出**：各线程将结果写入 `output[i]`。

### 3.3 为什么用 SIMT
每个元素的源地址 `index[i]` 不同，不连续也不规则，SIMD 无法批量寻址。SIMT 的每个线程独立计算地址，天然适配离散访存。950 的 SIMT 与 SIMD 共享同一 Vector Core 硬件，无需专用单元。

### 3.4 典型算子列表
Gather、Scatter、ConditionalSelect、SparseLookup、TopK（部分阶段）等。


### 3.5 离散访存的性能挑战
离散类算子的性能瓶颈通常在**访存带宽**而非计算算力：

- **缓存命中率低**：随机地址访问导致 Cache 命中率下降，大量请求穿透到 HBM。
- **内存事务合并困难**：相邻线程访问的地址可能相距甚远，无法合并为单次内存事务。
- **写冲突**：Scatter 类算子多线程写同一地址需原子操作，串行化降低并行度。

950 的 SIMT 通过大量线程并行隐藏访存延迟——当部分线程等待内存响应时，其他线程可继续执行。这种延迟隐藏策略与 GPU 类似，是处理离散访存的有效方法。


## 四、算子结构与硬件单元映射
| 算子结构 | 数据流 | 计算单元 | 编程模型 | 本章样例 |
|---|---|---|---|---|
| 连续矢量 | GM↔UB | Vector Core | SIMD | 2.5 C API Add |
| 矩阵 | GM→L1→L0A/L0B→L0C→GM | Cube Core | Tensor API | 2.6 Tensor MatMul |
| 离散矢量 | GM（直接寻址） | Vector Core | SIMT | 2.7 SIMT Gather |



> 下一节：[2.4 Ascend C 的 Hello World](02.04_hello_world.ipynb)
